# Compute per-site band quality baseline
* Downloads (if not already present) the 30 min low-tide, 0% cloud satellite images per site across 2016-2026
* Computes the per-site, per-band median/MAD baseline (see sentinel2.compute_and_save_band_quality_baseline)
* Only needs to be run once - the baseline is cached to utils.get_band_quality_baseline_path() and reused by sentinel2.flag_anomalous_band_dates

In [1]:
import geopandas
import pathlib
import dask.distributed

module_path = pathlib.Path.cwd().parent.parent / 'scripts'
import sys
if str(module_path) not in sys.path:
    sys.path.append(str(module_path))
import utils
import sentinel2

%load_ext autoreload
%autoreload 2

In [2]:
cluster = dask.distributed.LocalCluster()
client = dask.distributed.Client(cluster)
display(client)

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:61256/status,
Dashboard: http://127.0.0.1:61256/status,Workers: 4
Total threads: 8,Total memory: 31.73 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:61257,Workers: 0
Dashboard: http://127.0.0.1:61256/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:61276,Total threads: 2
Dashboard: http://127.0.0.1:61277/status,Memory: 7.93 GiB
Nanny: tcp://127.0.0.1:61260,


# Values to edit

In [3]:
all_training_sites =  ["Ihutai", "CatlinsLake", "CatlinsRiverMouth", "Childrens", "Duvauchelle", "Robinsons", "Takamatua", "Purau",
              "IveyBay_Nov25", "IveyBay_Feb26", "LeftBank_Nov25", "LeftBank_Feb26", "Paremata_Nov25", "Paremata_Feb26",
              "Paremata_Feb25", "ThePoint_Nov25", "ThePoint_Feb26", "Takapuwahia_Nov25", "Takapuwahia_Feb26", "IveyBay_ThePoint_LeftBank_Oct24"]

In [24]:
baseline_sites = all_training_sites
baseline_tide_deltas = {"30mins": {"hrs": 0, "mins": 30},
                        "45mins": {"hrs": 0, "mins": 45},
                        "1hr": {"hrs": 1, "mins": 0}}
baseline_tide_deltas = {"1hr": {"hrs": 1, "mins": 0}}
baseline_max_cloud_cover = 0
years = [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]

In [5]:
data_path = utils.get_data_path()
utils.create_data_folders()

# Cells to run
* Download the low-tide no-cloud images by year and site (skips sites already downloaded)
* Compute and save the per-site, per-band quality baseline

### Download satellite images
The 0% cloud, low-tide images across sites for each year and tide delta (30min/45min/1hr),
reusing the prediction-style per-year storage. Pooling across tide deltas (rather than dropping
the low-tide requirement entirely) grows the per-site sample size while keeping the baseline
restricted to genuinely low-tide reflectance.

In [6]:
satellite_paths = {}
for tide_delta_name, tide_delta in baseline_tide_deltas.items():
    _, satellite_path = utils.get_prediction_path(
        sample_method="sampling_2", method_2_threshold=0.5,
        model_max_cloud_cover=baseline_max_cloud_cover,
        predict_max_cloud_cover=baseline_max_cloud_cover,
        predict_low_tide_delta_hrs=tide_delta["hrs"],
        predict_low_tide_delta_mins=tide_delta["mins"],)
    satellite_paths[tide_delta_name] = satellite_path

In [7]:
satellite_paths

{'1hr': WindowsPath('C:/Local/repos/seagrass-detection/data/predictions/tide_1hrs_cloud_0/satellite_images')}

In [9]:
for tide_delta_name, tide_delta in baseline_tide_deltas.items():
    _, satellite_path = utils.get_prediction_path(
        sample_method="sampling_2", method_2_threshold=0.5,
        model_max_cloud_cover=baseline_max_cloud_cover,
        predict_max_cloud_cover=baseline_max_cloud_cover,
        predict_low_tide_delta_hrs=tide_delta["hrs"],
        predict_low_tide_delta_mins=tide_delta["mins"],
        satellite_only=True) 

    for baseline_site in baseline_sites:
        for year in years:
            print(f"{tide_delta_name} {baseline_site}: year {year}")

            satellite_file = satellite_path / f"{baseline_site}_{year}_sentinel-2.nc"
            site_polygon_file = utils.get_site_polygon_path(baseline_site)

            if (satellite_path / f"{baseline_site}_{years[-1]}_sentinel-2.nc").exists():
                print(f"\tAlready run for {baseline_site} as {baseline_site}_{years[-1]}_sentinel-2.nc exists. "
                      "Delete if you want redownloaded.")
                break

            if not satellite_file.exists():
                print(f"\tSave satellite image of the site {baseline_site} around lowtide without cloud in the year {year}")
                site_polygon = geopandas.read_file(site_polygon_file)
                for i in range(3):
                    try:
                        satellite_data = sentinel2.get_low_tide_no_cloud_images_in_year(
                            geometry=site_polygon,
                            max_cloud_cover=baseline_max_cloud_cover,
                            year=year,
                            low_tide_delta_hrs=tide_delta["hrs"],
                            low_tide_delta_mins=tide_delta["mins"],
                            anomaly_filter_baseline=None,
                        )
                        if len(satellite_data['time']) == 0:
                            print("Warning: No satellite data without cloud. Ignore")
                        else:
                            print("\t\tSave")
                            utils.write_netcdf_conventions_in_place(satellite_data)
                            satellite_data = satellite_data.rio.reproject(utils.CRS_NZTM)
                            satellite_data = satellite_data.rio.clip(site_polygon.geometry, drop=True, all_touched=True)
                            utils.write_netcdf_conventions_in_place(satellite_data)
                            utils.save_netcdf(satellite_data, satellite_file)
                        break
                    except Exception:
                        print("Caught exception downloading images and try again.")
                

1hr Ihutai: year 2016
	Already run for Ihutai as Ihutai_2026_sentinel-2.nc exists. Delete if you want redownloaded.
1hr CatlinsLake: year 2016
	Already run for CatlinsLake as CatlinsLake_2026_sentinel-2.nc exists. Delete if you want redownloaded.
1hr CatlinsRiverMouth: year 2016
	Already run for CatlinsRiverMouth as CatlinsRiverMouth_2026_sentinel-2.nc exists. Delete if you want redownloaded.
1hr Childrens: year 2016
	Already run for Childrens as Childrens_2026_sentinel-2.nc exists. Delete if you want redownloaded.
1hr Duvauchelle: year 2016
	Already run for Duvauchelle as Duvauchelle_2026_sentinel-2.nc exists. Delete if you want redownloaded.
1hr Robinsons: year 2016
	Already run for Robinsons as Robinsons_2026_sentinel-2.nc exists. Delete if you want redownloaded.
1hr Takamatua: year 2016
	Already run for Takamatua as Takamatua_2026_sentinel-2.nc exists. Delete if you want redownloaded.
1hr Purau: year 2016
	Already run for Purau as Purau_2026_sentinel-2.nc exists. Delete if you want

### Compute and save the band quality baseline
Pools all downloaded years across all tide deltas per site into one dataset, then computes
per-site, per-band median/MAD and caches it to utils.get_band_quality_baseline_path().

In [29]:
baseline = sentinel2.compute_and_save_band_quality_baseline(
    training_sites=baseline_sites,
    satellite_images_paths=list(satellite_paths.values()),
)
baseline

	Saved band quality baseline for 20 sites to C:\Local\repos\seagrass-detection\data\band_quality_baseline.csv


,site,band,median,mad,mad_raw,n_images
0,Ihutai,B04,506.00,65.423077,7.50,6
1,Ihutai,B03,646.50,69.347346,30.50,6
2,Ihutai,B02,569.00,66.711091,35.50,6
3,CatlinsLake,B04,351.00,64.400000,46.00,20
4,CatlinsLake,B03,374.50,70.000775,59.00,20
5,CatlinsLake,B02,265.50,63.037209,50.00,20
6,CatlinsRiverMouth,B04,799.00,84.818182,86.50,24
7,CatlinsRiverMouth,B03,877.00,83.455250,85.50,24
8,CatlinsRiverMouth,B02,636.00,65.670190,57.00,24
9,Childrens,B04,423.00,80.666667,80.00,64


In [30]:
import pandas
pandas.read_csv(r"C:\Local\repos\seagrass-detection\data\band_quality_baseline_tide_api.csv")

,site,band,median,mad,mad_raw,n_images
0,Ihutai,B04,502.00,51.494618,8.00,16
1,Ihutai,B03,642.00,59.765278,31.00,16
2,Ihutai,B02,568.00,53.913715,29.00,16
3,CatlinsLake,B04,356.00,57.154375,45.50,50
4,CatlinsLake,B03,378.00,63.650714,56.00,50
5,CatlinsLake,B02,262.00,52.527054,44.00,50
6,CatlinsRiverMouth,B04,814.00,93.410750,96.00,55
7,CatlinsRiverMouth,B03,877.00,85.140667,86.00,55
8,CatlinsRiverMouth,B02,637.00,50.491917,42.00,55
9,Childrens,B04,423.00,82.479365,82.00,159
